# Topic 12 — Decision Trees
### Theory → tiny example → from-scratch splitting logic → sklearn → overfitting demo.

A decision tree predicts by asking a series of yes/no questions about features, splitting the data
at each **node** until it reaches a **leaf** with a final prediction.

```text
                [feature_x <= 5?]
               /                \
            Yes                  No
             |                    |
      [feature_y <= 2?]        leaf: class 1
       /            \
    leaf: 0        leaf: 1
```

- **Node**: a decision point (a question about one feature).
- **Split**: how the data at a node gets divided into two branches.
- **Leaf**: the end of a branch — the final predicted class.
- **Decision rule**: the condition at a node (e.g. `feature_x <= 5`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)

## 1. Gini impurity & entropy — measuring how "mixed" a node is

Both measure impurity: 0 = a node is perfectly pure (all one class), higher = more mixed.
The tree picks whichever split reduces impurity the most.

```text
Gini(node)    = 1 - sum(p_i^2)              for each class i
Entropy(node) = -sum(p_i * log2(p_i))       for each class i
```

In [ ]:
def gini(labels):
    labels = np.array(labels)
    if len(labels) == 0:
        return 0
    probs = np.array([np.mean(labels == c) for c in np.unique(labels)])
    return 1 - np.sum(probs ** 2)

def entropy(labels):
    labels = np.array(labels)
    if len(labels) == 0:
        return 0
    probs = np.array([np.mean(labels == c) for c in np.unique(labels)])
    probs = probs[probs > 0]   # avoid log(0)
    return -np.sum(probs * np.log2(probs))

pure_node = [1, 1, 1, 1]
mixed_node = [0, 1, 0, 1]
somewhat_mixed = [0, 0, 0, 1]

for name, node in [("pure", pure_node), ("mixed 50/50", mixed_node), ("mostly one class", somewhat_mixed)]:
    print(f"{name}: gini={gini(node):.3f}, entropy={entropy(node):.3f}")
# Pure node -> 0 impurity. 50/50 mixed -> maximum impurity.

## 2. Information gain — choosing the best split

**Information gain** = impurity before the split - weighted impurity after the split.
The tree tries every possible feature/threshold and picks the split with the highest information gain.

In [ ]:
def information_gain(parent_labels, left_labels, right_labels, criterion=gini):
    n = len(parent_labels)
    n_left, n_right = len(left_labels), len(right_labels)
    weighted_child_impurity = (n_left/n) * criterion(left_labels) + (n_right/n) * criterion(right_labels)
    return criterion(parent_labels) - weighted_child_impurity

# Toy dataset: 1 feature, split on feature <= 5
X_toy = np.array([1, 2, 3, 6, 7, 8])
y_toy = np.array([0, 0, 0, 1, 1, 1])   # perfectly separable at threshold=5

for threshold in [2, 5, 7]:
    left_mask = X_toy <= threshold
    gain = information_gain(y_toy, y_toy[left_mask], y_toy[~left_mask])
    print(f"split at x<={threshold}: information gain = {gain:.3f}")
# threshold=5 gives the highest gain -- it's the split that perfectly separates the two classes.

## 3. sklearn's DecisionTreeClassifier

In [ ]:
X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.3, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

print("train accuracy:", accuracy_score(y_train, tree.predict(X_train)))
print("test accuracy:", accuracy_score(y_test, tree.predict(X_test)))

plt.figure(figsize=(12, 6))
plot_tree(tree, feature_names=["feature_0", "feature_1"], class_names=["class0", "class1"],
          filled=True, rounded=True, fontsize=9)
plt.title("Decision tree structure (max_depth=3)")
plt.show()
# Each box shows: the split rule, the gini impurity, the number of samples, and the class distribution.

## 4. Why trees overfit — the role of `max_depth`

An unrestricted tree can keep splitting until every leaf is perfectly pure — including memorizing
noise. This is classic overfitting (same concept as Topic 5): great train accuracy, poor test accuracy.
**Pruning** (limiting depth, or removing branches after the fact) fights this.

In [ ]:
depths = [1, 3, 5, None]   # None = unlimited depth
train_accs, test_accs = [], []

for depth in depths:
    t = DecisionTreeClassifier(max_depth=depth, random_state=42)
    t.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, t.predict(X_train)))
    test_accs.append(accuracy_score(y_test, t.predict(X_test)))

depth_labels = [str(d) for d in depths]
x_pos = np.arange(len(depths))

plt.figure(figsize=(6, 4))
plt.plot(x_pos, train_accs, marker="o", label="train accuracy")
plt.plot(x_pos, test_accs, marker="o", label="test accuracy")
plt.xticks(x_pos, depth_labels)
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.title("Deeper trees: train accuracy keeps rising, test accuracy plateaus/drops")
plt.legend()
plt.show()
# Watch the gap between the two lines widen as depth increases -- that gap IS overfitting.

## 5. Decision boundary at different depths

In [ ]:
def plot_tree_boundary(ax, X, y, max_depth):
    t = DecisionTreeClassifier(max_depth=max_depth, random_state=42).fit(X, y)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = t.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="bwr")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k")
    ax.set_title(f"max_depth={max_depth}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, depth in zip(axes, [1, 3, None]):
    plot_tree_boundary(ax, X, y, depth)
plt.tight_layout()
plt.show()
# Notice trees produce BLOCKY, axis-aligned boundaries (unlike logistic regression's smooth line)
# -- because every split only looks at ONE feature at a time.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Implement your own function `best_split(X_column, y)` that tries every unique value in X_column
#    as a threshold and returns the one with the highest information gain (using the functions above).
# 2. Change DecisionTreeClassifier's criterion parameter from "gini" to "entropy" and compare test accuracy.
# 3. Try min_samples_leaf=10 instead of max_depth as a different way to control overfitting --
#    plot its decision boundary too.
# 4. Look at feature_importances_ (tree.feature_importances_) -- which feature mattered more?

---
### Next up: **Topic 13 — Random Forest** (an ensemble of many decision trees).

Say "next" when you're ready.